In [34]:
import spacy
from geopy.geocoders import Nominatim

# Load SpaCy model, figures out what part of the text is about a location
ner_model = spacy.load("en_core_web_sm")

# Geocode with Nominatim, maps names of locations to their coordinates
geolocator = Nominatim(user_agent="geo_demo")

In [43]:
examples = ["There is a problem in Dallas, TX",
            "Another tweet about Dallas TX no comma this time",
            "Yet a third post about Dallas, Texas",
            "What if talk about 2 places? I would like to visit Austin TX, but I want to focus on Houston TX",
            "I am currently visting family in downtwon LOS angeles!",
            "This tweet has no location",
            "I wonder what I should eat for dinner",
            "north cali "]


In [44]:

coordinates = []
# Extract the location subject of the text, if there is any. Note that some places are amiguous and might be mistaken
# for similarly named places (see last example, a human could tell what the text is about, but not the model, currently)
for i, example in enumerate(examples):
  doc = ner_model(example)
  locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]

  print("="*50)
  print(f"EXAMPLE {i}: {example}")

  print("Locations found:", locations)



  for loc in locations:
    geo = geolocator.geocode(loc)
    if geo:
        print(f"{loc} → ({geo.latitude}, {geo.longitude})")
        coordinates.append([geo.latitude, geo.longitude])
    else:
      print("No location found in text")

EXAMPLE 0: There is a problem in Dallas, TX
Locations found: ['Dallas']
Dallas → (32.7762719, -96.7968559)
EXAMPLE 1: Another tweet about Dallas TX no comma this time
Locations found: ['Dallas']


Dallas → (32.7762719, -96.7968559)
EXAMPLE 2: Yet a third post about Dallas, Texas
Locations found: ['Dallas', 'Texas']
Dallas → (32.7762719, -96.7968559)
Texas → (31.2638905, -98.5456116)
EXAMPLE 3: What if talk about 2 places? I would like to visit Austin TX, but I want to focus on Houston TX
Locations found: ['Houston']
Houston → (29.7589382, -95.3676974)
EXAMPLE 4: I am currently visting family in downtwon LOS angeles!
Locations found: ['LOS angeles']
LOS angeles → (34.0536909, -118.242766)
EXAMPLE 5: This tweet has no location
Locations found: []
EXAMPLE 6: I wonder what I should eat for dinner
Locations found: []
EXAMPLE 7: north cali 
Locations found: []


In [45]:
 coordinates

[[32.7762719, -96.7968559],
 [32.7762719, -96.7968559],
 [32.7762719, -96.7968559],
 [31.2638905, -98.5456116],
 [29.7589382, -95.3676974],
 [34.0536909, -118.242766]]

Note some messy / abreviated text is not easily recognized (last example). Now to visualize the above's outputs

In [38]:
# Python library based on leaflet.js, same tool we are using on the frontend
!pip install folium

import folium
from geopy.geocoders import Nominatim

from folium.plugins import MarkerCluster  # helps group pins when they are about the same place
# ex: two tweets about Dallas will result in pins directly on top of each other, making one inaccessible

In [46]:
# Create a base map centered at first location
map_ = folium.Map(location=[coordinates[0][0], coordinates[0][1]])

marker_cluster = MarkerCluster().add_to(map_)

for lat, lon in coordinates:
    folium.Marker([lat, lon]).add_to(marker_cluster)
    folium.Marker()


map_